In [1]:
!pip install -q -U google-genai sentence-transformers chromadb langchain-text-splitters pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.1/388.1 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

In [2]:
import os
from google.colab import userdata, files
from google import genai
from sentence_transformers import SentenceTransformer
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader

In [4]:
# 1. API key setup

api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("G_A_K")

if not api_key:
    try:
        api_key = userdata.get("Vaishnavi")
    except Exception:
        pass

if api_key:
    client = genai.Client(api_key=api_key)
else:
    raise ValueError("Gemini API Key was not found in Colab Secrets or Environment Variables.")

In [5]:
# 2. Upload and read PDF files

print("Please Upload one or more PDF Files:")

uploaded = files.upload()

pdf_texts = []

for filename in uploaded.keys():

    if filename.endswith(".pdf"):

        reader = PdfReader(filename)

        text = ""

        for page_num, page in enumerate(reader.pages):

            page_text = page.extract_text()

            if page_text:
                text += f"\n--- Page {page_num + 1} ---\n"
                text += page_text

        if text.strip():
            pdf_texts.append(text)
            print(f"Loaded '{filename}' ({len(reader.pages)} pages).")

if not pdf_texts:
    raise ValueError("No Valid PDF files Uploaded. Please re-run and upload a .pdf file.")

Please Upload one or more PDF Files:


Saving VaishnaviSardar_Resume.pdf to VaishnaviSardar_Resume.pdf
Loaded 'VaishnaviSardar_Resume.pdf' (2 pages).


In [6]:
# 3. Chunk text

full_pdf_content = "\n\n".join(pdf_texts)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_text(full_pdf_content)

print(f"Extracted and split document into {len(chunks)} text chunks.")

Extracted and split document into 7 text chunks.


In [7]:
# 4. Generate embeddings and store in ChromaDB

print("Loading embedding model and building vector index...")

embedder = SentenceTransformer("all-MiniLM-L6-v2")

chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection(name="pdf_rag_collection")
except Exception:
    pass

collection = chroma_client.create_collection(name="pdf_rag_collection")

chunk_embeddings = embedder.encode(chunks).tolist()

chunk_ids = [f"doc_chunk_{i}" for i in range(len(chunks))]

collection.add(
    documents=chunks,
    embeddings=chunk_embeddings,
    ids=chunk_ids
)

print("Successfully indexed", len(chunks), "chunks into 'pdf_rag_collection'.")

Loading embedding model and building vector index...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Successfully indexed 7 chunks into 'pdf_rag_collection'.


In [8]:
# 5. Retrieval Function

def retrieve_pdf_context(query: str, top_k: int = 3):

    query_embedding = embedder.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )

    return results["documents"][0]

In [9]:
# 6. Ask Gemini

def ask_pdf(query: str):

    context_passages = retrieve_pdf_context(query)

    context_str = "\n".join(f"- {p}" for p in context_passages)

    prompt = f"""You are an intelligent document analysis assistant.

Answer the question using the provided context.

If the information is not contained within the provided context, state clearly:

"I cannot find this information in the document."

PDF Context:
{context_str}

Question:
{query}

Answer:
"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return response.text, context_passages

In [ ]:
import time

print("=" * 60)
print("PDF CHATBOT READY! Type your question below (or type 'exit' to quit).")
print("=" * 60)

while True:

    user_query = input("\nAsk a question about your PDF: ").strip()

    if user_query.lower() in ["exit", "quit", "q"]:
        print("Exiting PDF Chatbot. Goodbye!")
        break

    if not user_query:
        continue

    # Retry if Gemini temporarily returns 503
    max_retries = 3

    for attempt in range(max_retries):
        try:
            answer, context = ask_pdf(user_query)
            break

        except Exception as e:

            error_message = str(e)

            if "503" in error_message or "UNAVAILABLE" in error_message:

                if attempt < max_retries - 1:
                    wait_time = 3 * (attempt + 1)

                    print(
                        f"\nGemini server is temporarily busy."
                        f"\nRetrying in {wait_time} seconds..."
                    )

                    time.sleep(wait_time)

                else:
                    print("\nGemini is currently unavailable.")
                    print("Please try your question again after some time.")
                    answer = None
                    context = []

            else:
                print("\nAn error occurred:")
                print(error_message)
                answer = None
                context = []
                break

    # Don't print snippets if retrieval failed
    if answer is None:
        continue

    print("\n--- RETRIEVED PDF SNIPPETS ---")

    if context:
        for i, snippet in enumerate(context, 1):
            print(f"[{i}] {snippet[:150]}...")
    else:
        print("No PDF snippets retrieved.")

    print("\n--- GEMINI RESPONSE ---")
    print(answer)

    print("-" * 60)

PDF CHATBOT READY! Type your question below (or type 'exit' to quit).

Ask a question about your PDF: Give a summary of the document.

Gemini server is temporarily busy.
Retrying in 3 seconds...

Gemini server is temporarily busy.
Retrying in 6 seconds...

Gemini is currently unavailable.
Please try your question again after some time.

Ask a question about your PDF: Give a summary of the document.

Gemini server is temporarily busy.
Retrying in 3 seconds...

--- RETRIEVED PDF SNIPPETS ---
[1] Database & Analytics: SQL (Basic), Excel 
Power BI (Learning)  
Tools & Technologies: Visual Studio Code, 
Git & GitHub, Microsoft Office Suite, R 
st...
[2] 2. Student Management System 
 
 Developed a console-based localized system to safely record, lookup, and manage student detail databases.  
 Technolo...
[3] MCA | Sterling Institute of Management Studies and Research, Navi Mumbai | 2025 – 27 | Pursuing   
B.Sc. | Computer Science | Brijlal Biyani Science C...

--- GEMINI RESPONSE ---
Based 